In [0]:
from pyspark.sql import functions as F

REF = "/Volumes/voebem/bronze/arquivos/referencias"

#caractere que nao existe no arquivo --> desliga o quoting do leitor de CSV
SEM_ASPAS = chr(0)

In [0]:
#leitura e configuração do arquivo
aerodromos = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", "1")
    .option("encoding", "ISO-8859-1") #latin-1, não UTF-8
    .option("quote", SEM_ASPAS) #desliga o quoting: aspas aqui são segundos
    .load(f"{REF}/AerodromosPublicos.csv") #carregamos da origem
)

#dando apelido a colunas com select, ao invés de dicionário, como feito no caso da basta vra, pra tirar o espaço.
aerodromos = aerodromos.select(
    F.col("`Código OACI`").alias("icao"),
    F.col("CIAD").alias("ciad"),
    F.col("Nome").alias("nome"),
    F.col("`Município`").alias("municipio"),
    F.col("UF").alias("uf"),
    F.col("`Município Servido`").alias("municipio_servido"),
    F.col("`UF Servido`").alias("uf_servido"),
    F.col("Latitude").alias("latitude"),
    F.col("Longitude").alias("longitude"),
    F.col("Altitude").alias("altitude"),
    F.col("`Situação`").alias("situacao"),
).withColumn("_ingerido_em", F.current_timestamp())

#escrita do arquivo
aerodromos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("voebem.bronze.aerodromos") #--> destino

#select para vermos como tá o dataset
print(f"bronze.aerodromos: {spark.table('voebem.bronze.aerodromos').count():,}linhas")
display(spark.sql("""
        SELECT icao,
               nome,
               municipio,
               uf
        FROM voebem.bronze.aerodromos
        WHERE icao IN ('SBRB', 'SBGR', 'SBSP', 'SBFZ')
                 """)
        )


In [0]:
#temos tabelas de origem de coisas similares --> uma é pra empresas nacionais e outra para internacionais. Vamos manter isso na bronze do jeito que elas são. tratamento pode ser feito na prata
def ler_empresas(arquivo: str):
    """Le um cadastro de empresas. Sem uniao, sem enriquecimento: uma tabela por arquivo."""
    return (
        spark.read.format("csv")
        .option("sep", ";")
        .option("header", "true")
        .option("skipRows", 1)
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .load(f"{REF}/{arquivo}")
        .select(
            F.col("ICAO").alias("icao"),
            F.col("Estrangeira").alias("sigla_iata"),
            F.col("Razao").alias("razao_social"),
            F.col("Servico").alias("servico"),
            F.col("Cidade").alias("cidade"),
            F.col("UF").alias("uf"),
            F.col("Ativa").alias("situacao"),
        )
        .withColumn("_arquivo_origem", F.lit(arquivo))
        .withColumn("_ingerido_em", F.current_timestamp())
    )

In [0]:
#salvando uma tabela para cada caso: estrangeira e nacional
for arquivo, tabela in [
    ("pda_empresas_aereas_nacionais.csv",    "voebem.bronze.empresas_nacionais"),
    ("pda_empresas_aereas_estrangeiros.csv", "voebem.bronze.empresas_estrangeiras"),
]:
    ler_empresas(arquivo).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(tabela)
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

In [0]:
#feita a ingestão das tabelas, nas próximas 3 células vamos verificar como estas tabelas estão:
#primeiro usaremos sql para ver quantas linhas cada tabela tem e quantas linhas a coluna icao não está nula em cada uma delas
display(spark.sql("""
    SELECT 'empresas_nacionais' AS tabela, COUNT(*) AS linhas,
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
    FROM voebem.bronze.empresas_nacionais
    UNION ALL
    SELECT 'empresas_estrangeiras', COUNT(*),
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END)
    FROM voebem.bronze.empresas_estrangeiras
"""))

In [0]:
#agora veremos a tabela de empresas nacionais, filtrando no where para vermos somente as linhas que correspondem a empresas nacionais
display(spark.sql("""
    SELECT icao, razao_social, servico, uf, situacao
    FROM voebem.bronze.empresas_nacionais
    WHERE icao IN ('GLO','TAM','AZU','PAM')
    ORDER BY icao
    """))

#agora vamos ver a tabeça de empresas estrangeiras filtrando no where para vermos somente as linhas que correspondem a empresas estrangeiras
display(spark.sql("""
    SELECT icao, razao_social, servico, situacao
    FROM voebem.bronze.empresas_estrangeiras
    WHERE icao IN ('AAL','TAP','AVA','ARG')
    ORDER BY icao
    """))


In [0]:
#aqui é a parte de código, que é um dado interno da própria ANAC que vamos usar para nos encontrar. Esses códigos identificam se é uma linha doméstica, se é internacional, etc. Nos ajuda a entender os detalhes de negócio e compreender o contexto do dado para fazermos a análise.
CODIGOS = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira"),
]

codigos = spark.createDataFrame(CODIGOS, "dominio string, codigo string, descricao string")
codigos.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("voebem.bronze.codigos_operacao")

print(f"bronze.codigos_operacao: {spark.table('voebem.bronze.codigos_operacao').count()} linhas")
display(spark.table("voebem.bronze.codigos_operacao"))


In [0]:
#agora vamos conseguir ver todo nosso catálogo da camada bronze. aqui já temos toda a ingestão dos dados que precisamos pra fazer a leitura, tá construída
display(spark.sql("SHOW TABLES IN voebem.bronze"))

In [0]:
#o que o delta guardou mesmo sem a gente pedir. ele faz versionamento automaticamente
display(spark.sql("""
    SELECT version, timestamp, operation,
           operationMetrics.numOutputRows AS linhas_escritas
    FROM (DESCRIBE HISTORY voebem.bronze.vra)
    ORDER BY version
"""))

In [0]:
#a versão 0 é a primeira carga do marco-03; a carga seguinte é a segunda execução da ingestão (aquela que provou a idempotência). dá para consultar as duas lado a lado.
display(spark.sql("""
    SELECT 'versao 0 (1a carga)'   AS versao,
           COUNT(*)                AS linhas,
           MIN(_ingerido_em)       AS ingerido_em
    FROM voebem.bronze.vra VERSION AS OF 0
    UNION ALL
    SELECT 'versao atual', COUNT(*), MIN(_ingerido_em)
    FROM voebem.bronze.vra
"""))


In [0]:

#Mesmo número de linhas, `_ingerido_em` diferente: a prova de idempotência do marco anterior, agora reconstruída **do histórico**, sem ter guardado nada.
# isso é propriedade do formato de tabela aberto, não código nosso. Todo `write` no Delta grava um commit no log de transações; o dado antigo continua nos arquivos Parquet até um `VACUUM`. Auditoria e rollback saem de graça.
for tabela, comentario in [
    ("voebem.bronze.aerodromos",
     "Bronze - cadastro de aerodromos publicos da ANAC, como chegou. Chave: codigo ICAO (OACI). "
     "Cobre apenas aerodromos brasileiros - aeroportos estrangeiros do VRA nao estao aqui."),
    ("voebem.bronze.empresas_nacionais",
     "Bronze - cadastro de empresas aereas NACIONAIS da ANAC, como chegou. Chave: codigo ICAO. "
     "Nao unir com empresas_estrangeiras nesta camada: a uniao e feita na silver."),
    ("voebem.bronze.empresas_estrangeiras",
     "Bronze - cadastro de empresas aereas ESTRANGEIRAS autorizadas a operar no Brasil, como chegou. "
     "Chave: codigo ICAO. Cadastro separado do nacional na origem, mantido separado no bronze."),
    ("voebem.bronze.codigos_operacao",
     "Bronze - seed table curada a partir da pagina de descricao de variaveis da ANAC. "
     "Traduz codigo_di e codigo_tipo_linha para descricao em portugues."),
]:
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")

print("comentarios aplicados")